In [1]:
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex":True,
    "font.family":"Helvetica",
    "font.size"  : 13,
})

# import equivariant steerable cnn
import escnn.gspaces
import escnn.nn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import from_numpy
from sklearn.model_selection import train_test_split

from tqdm import tqdm
# Checking if you have a GPU and CUDA set up -- if not, training will take longer by running on the CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.empty_cache()
# device = 'cpu'
# Set the plotting style
#plt.style.use('seaborn-poster')

from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from os import listdir
from os.path import isfile, join

from copy import deepcopy

from getdist import MCSamples, plots

import pickle as pk

from hyperparameters import hparams

In [2]:
ens = 20 # Turn this up
path = '/project/ag-weller/Kai.Lehman/gaussian_fields/'
onlyfiles = [f[:-4] for f in listdir(path) if f[-3:] == 'npy']
onlyfiles_250_by_250 = np.array([file for file in onlyfiles if '250_by_250' in file])

# subsample
# 20x2000 takes up 45755MiB
print(np.shape(onlyfiles_250_by_250))
onlyfiles_250_by_250 = np.random.choice(onlyfiles_250_by_250, 100, replace=False)


#onlyfiles_split = np.array([s.split('_') for s in onlyfiles])
onlyfiles_250_by_250_split = np.array([s.split('_') for s in onlyfiles_250_by_250])
#onlyfiles_250_by_250 = np.array([file for file in onlyfiles_split if (file[3]=='250') & (file[5]=='250')])
print(onlyfiles_250_by_250.shape)

(10000,)
(100,)


In [3]:
stds = onlyfiles_250_by_250_split[:,8].astype(np.float32)
corrlens = onlyfiles_250_by_250_split[:,10].astype(np.float32)

#imgs = np.zeros((ens*len(onlyfiles),100,100))
#imgs = np.zeros((ens*len(onlyfiles_250_by_250),250,250))
imgs = np.zeros((len(onlyfiles_250_by_250),ens,250,250),dtype='float32')
for i, name in enumerate(onlyfiles_250_by_250):
    img_stack = np.load(path+name+'.npy')
    #imgs[i*ens:(i+1)*ens] = img_stack[:ens]
    imgs[i] = np.asarray(img_stack,dtype='float32')
    if i%500==0: print(i)

0


In [4]:
thetas = np.concatenate((np.atleast_2d(stds),np.atleast_2d(corrlens)),axis=0).T
print(np.shape(onlyfiles_250_by_250_split))
print(np.shape(thetas))
print(np.shape(img_stack))

(100, 11)
(100, 2)
(20, 250, 250)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(imgs, thetas, test_size=0.1, random_state=42)
#X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

# now unravel
X_train = from_numpy(X_train.reshape(len(X_train)*ens,1,250,250)).to(device)
#X_val = from_numpy(X_val.reshape(len(X_val)*ens,1,250,250)).to(device)
X_test = from_numpy(X_test.reshape(len(X_test)*ens,1,250,250)).to(device)

In [6]:
y_train_tiled = from_numpy(np.repeat(y_train,repeats=ens,axis=0)).to(device)
#y_val_tiled = from_numpy(np.repeat(y_val,repeats=ens,axis=0)).to(device)
y_test_tiled = from_numpy(np.repeat(y_test,repeats=ens,axis=0)).to(device)

# building one theta vector
#theta = from_numpy(np.vstack((stds_tiled,corrlens_tiled)).T).to('cuda')
print(y_train_tiled.shape)

torch.Size([1800, 2])


In [7]:
# set up C3NN
bandlimit = 6
gspace = escnn.gspaces.rot2dOnR2(N=-1, maximum_frequency=bandlimit)


""" Implements the non-linear convolution operation using the recursive formula."""
class NonlinearConv(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order):
        super().__init__()
        self.order = order
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size

        """
        # Initialize filter similar as to how Pytorch initializes nn.Conv2d, but all positive
        # This is the old way of initializing C3NN filters without rotational equivariance/invariance
        weight_sample_k = 1.0 / np.sqrt(filter_size**2 * in_channels)
        init_filt = (2 * weight_sample_k * torch.rand(num_filts, in_channels, filter_size, filter_size, dtype=torch.float32))
        self.conv_filt = torch.nn.Parameter(init_filt)
        # Initialize some slices of the filter to be 0
        with torch.no_grad():
            self.conv_filt[:, :, int(self.filter_size/2)-2:int(self.filter_size/2)+3, int(self.filter_size/2)-2:int(self.filter_size/2)+3]=0.0
        """
        
        # Initialize the escnn filter
        feat_type_in = escnn.nn.FieldType(gspace, self.in_channels*[gspace.trivial_repr])
        feat_type_out = escnn.nn.FieldType(gspace, self.num_filts*[gspace.trivial_repr])
        # THIS LAYER KILLS DEEPCOPY
        self.conv = escnn.nn.R2Conv(in_type=feat_type_in, out_type=feat_type_out, bias=False, kernel_size=self.filter_size) 
        
        # to enable deepcopy
        self.detached = False
        
    def forward(self, x):
        # We will work with strictly positive filters for easier interpretation!
        # foll
        if self.detached == False:
            self.abs_filt = self.conv.expand_parameters()[0].abs()
        elif self.detached == True:
            pass
        
        # Convolutions of powers of the input with powers of the absolute valued filter
        conv_powers = [F.conv2d(x ** k, self.abs_filt ** k, stride=1, padding=int((self.filter_size-1)/2.0)) for k in range(1, self.order + 1)]
        
        #troubleshooting:
        #conv_powers = [self.convnormal(x ** k) for k in range(1, self.order + 1)]
        
        # C_0 is defined to be one everywhere
        corrs = [torch.ones_like(conv_powers[0])]

        # Compute each order of C_n recursively
        for n in range(1, self.order + 1):
            Cn = torch.zeros_like(conv_powers[0])
            for k in range(n):
                Cn += (-1) ** k * conv_powers[k] * corrs[n - k - 1]
            Cn /= n
            corrs.append(Cn)

        # Stack correlator maps along the channel dimension, dropping the zeroth order (C_0)
        # Here the channel dimension refers to the number of channels
        #return torch.cat(corrs[1:], dim=-3)
        return torch.cat(corrs[1:], dim=-3)

    def _detach(self):
        del self.conv
        self.abs_filt = self.abs_filt.detach()
        self.detached = True


""" Does a simple average over the last two dimensions of the input."""
class Mean2d(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return x.mean((-2, -1))


""" The full CCNN is Input -> NonlinearConv -> Spatial Avg -> BatchNorm -> Logistic. The way it is constructed
    here assumes a binary classification task, but this can be extended to the multiclass case."""
class CCNN(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order, absbeta=False, slice=False, pool=False):
        super().__init__()
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size
        self.order = order
        self.absbeta = absbeta

        # the model architecture with escnn
        self.corr_conv = NonlinearConv(self.in_channels, self.num_filts, self.filter_size, self.order)
        self.spatial_mean = Mean2d()
        # I preserve the batch normalization layer here so that we can scale the compressed summary statistics to the same order of magnitude
        # This may help the numerical stability of inference in the 2nd part
        self.bn = nn.BatchNorm1d(num_filts * order, affine=True, eps=3.185289e-35, dtype=torch.float32)


        #troubleshooting
        #self.conv = Conv2d(1,1,11)
    
    def forward(self, x, chop=True):    
        
        x = self.corr_conv(x)
        #x = self.conv(x)
        x = self.spatial_mean(x)
        if chop == False:
            x = self.bn(x)


        return x

    def detach(self):
        self.corr_conv._detach()


#=== 2) Set up some things for training ===#
# Number of times the model will see the full dataset
#epochs = 5
# The initial learning rate -- an important hyperparameter
#lr = 0.505631 
#lr = 0.5
# The strength of the regularization loss -- we can make this very strong in these easy toy examples
#gamma = 0.030383
# The number of input channels 
in_channels = 1
# The number of filters to learn in our model
num_filts = 1
# The spatial length of the filters
filter_size = 250
# The highest order of correlation measured
order = 2
# Make our model -- for simplicity, let's train a two-filter model

In [8]:
X_train.shape

torch.Size([1800, 1, 250, 250])

In [9]:
print(stds.min(),stds.max())
print(corrlens.min(),corrlens.max())

1.0 50.49495
5.0 22.818182


In [ ]:
ccnn = CCNN(in_channels=in_channels, num_filts=num_filts, filter_size=filter_size, order=order, absbeta=False).to(device='cuda')

# Set up prior for sbi
lower = np.array([1.,5.])
upper = np.array([51.,23.])
#lower = np.array([.8,3.])
#upper = np.array([4.,30.])
prior = utils.BoxUniform(low=from_numpy(lower), high=from_numpy(upper),device='cuda')


#set up sbi architecture
density_estimator_custom = posterior_nn(model="maf", hidden_features=hparams.flow_hidden_features, num_transforms=hparams.flow_num_transforms, device='cuda', embedding_net=ccnn)
inference = SNPE_C(prior, device='cuda', density_estimator=density_estimator_custom)

# joint training
density_estimator = inference.append_simulations(y_train_tiled, X_train).train(learning_rate=hparams.learning_rate,training_batch_size=hparams.flow_training_batch_size,max_num_epochs=2)
# DANGER DANGER
#ccnn.detach()
# now deepcopy works:
test = deepcopy(density_estimator._embedding_net)

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(
/home/k/Kai.Lehman/C3NN_inference/sbi/utils/user_input_checks.py:435: UserWarning: Mismatch between the device of the data fed to the embedding_net and the device of the embedding_net's weights. Fed data has device 'cpu' vs embedding_net weights have device 'cuda:0'. Automatically switching the embedding_net's device to 'cpu', which could otherwise be done manually using the line `embedding_net.to('cpu')`.
  warnings.warn(


 Training neural network. Epochs trained: 3DANGER!!!! No more deepycopy after training!!!


In [ ]:
with open(f"inference_2000x20sub_filter250.pkl", "wb") as handle:
            pk.dump(inference, handle)
with open(f"density_estimator_2000x20sub_filter250.pkl", "wb") as handle:
            pk.dump(deepcopy(density_estimator), handle)
print('Model saved')

In [12]:
sys.exit()

NameError: name 'sys' is not defined

In [130]:
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter101_order_2_density_estimator.pkl', 'rb') as f:
    density_estimator = pk.load(f)
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter101_order_2_inference.pkl', 'rb') as f:
    inference = pk.load(f)
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter101_order_3_density_estimator.pkl', 'rb') as f:
    density_estimator3 = pk.load(f)
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter101_order_3_inference.pkl', 'rb') as f:
    inference3 = pk.load(f)

In [35]:
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter150_density_estimator.pkl', 'rb') as f:
    density_estimator = pk.load(f)
with open('/project/ag-weller/Kai.Lehman/C3NN_Models/lr_1.00e-04_features_64_transforms_3_batchsize_32_2xfilter150_inference.pkl', 'rb') as f:
    inference = pk.load(f)

In [131]:
train_loader, val_loader = inference.get_dataloaders(
            #start_idx,
            training_batch_size=1,
            #validation_fraction,
            resume_training=True,
            #dataloader_kwargs=dataloader_kwargs,
            dataloader_kwargs={"drop_last": False},
)

theta_test, img_test, _ = next(iter(val_loader))

In [132]:
train_loader3, val_loader3 = inference3.get_dataloaders(
            #start_idx,
            training_batch_size=1,
            #validation_fraction,
            resume_training=True,
            #dataloader_kwargs=dataloader_kwargs,
            dataloader_kwargs={"drop_last": False},
)


In [75]:
# build posterior and sample
index = 2000
iterator = iter(val_loader)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
theta_test, img_test, _ = next(iterator)
posterior = inference.build_posterior(deepcopy(density_estimator))
posterior_samples = posterior.sample((1000000,), x=img_test).cpu()

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


Drawing 1000000 posterior samples:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [76]:
names = np.array(['\sigma','\lambda'])
params = theta_test[0].cpu()


samples = MCSamples(samples=posterior_samples.numpy(), names=names, labels=names, settings={'ignore_rows':.2},label='sbi, Likelihood-Free')
g = plots.get_subplot_plotter()
#g.settings.lab_fontsize = 17
#g.settings.axes_fontsize = 14.3
#g.settings.legend_fontsize = 14.5
g.triangle_plot([samples], names, title_limit=1, filled=True,
                     markers={'\sigma':params[0],'\lambda':params[1]},
                     marker_args={'c':'black'})

plt.show()
plt.savefig('cambridge_2000x20_2xfilter101_order_2.pdf')

Removed 0.2 as burn in


In [125]:
# let's do a more statistical test over the entire validation set

no_val_sims = 50
posterior_samples_all = np.zeros((no_val_sims,100000,2))
params = np.zeros((no_val_sims,2))
c= 0 
for i, (param, observation, _) in enumerate(val_loader):
    if param.cpu()[0,0] < 20 and param.cpu()[0,1] < 11:
        params[c] = param.cpu()
        observation.to('cuda')
        posterior_samples_all[c] = posterior.sample((100000,), x=observation).cpu()
        c += 1
        if i%10 == 0:
            print(i)
        if c == no_val_sims-1:
            break

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

... (41 outputs left out)

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

In [92]:
print(np.shape(posterior_samples_all))
print(np.shape(params))

(50, 100000, 2)
(50, 2)


In [104]:
plt.clf()
fig, ax = plt.subplots(2)
parts0 = ax[0].violinplot(posterior_samples_all[:no_val_sims,:,0].T-params[:no_val_sims,0],params[:no_val_sims,0],showextrema=False,quantiles=np.tile([.16,.84],(no_val_sims,1)).T,widths=.5)
parts1 = ax[1].violinplot(posterior_samples_all[:no_val_sims,:,1].T-params[:no_val_sims,1],params[:no_val_sims,1],showextrema=False,quantiles=np.tile([.16,.84],(no_val_sims,1)).T,widths=.3)

for i, axis in enumerate(ax):
    axis.axhline(0,color='black')
ax[0].set_xlabel(r'$\sigma_\mathrm{True}$')
ax[0].set_ylabel(r'$\sigma_\mathrm{True} - \sigma_\mathrm{Pred}$')
ax[1].set_xlabel(r'$\lambda_\mathrm{True}$')
ax[1].set_ylabel(r'$\lambda_\mathrm{True} - \lambda_\mathrm{Pred}$')

ax[0].set_ylim([-5,5])
ax[1].set_ylim([-5,5])
ax[1].set_xlim([4.5,11.])

plt.tight_layout()
plt.savefig('cambridge_violin_2000x20_2xfilter101_order_2.pdf')

In [129]:
names = np.array(['\sigma','\lambda'])
ind = 23

samples = MCSamples(samples=posterior_samples_all[ind], names=names, labels=names)
g = plots.get_subplot_plotter()
#g.settings.lab_fontsize = 17
#g.settings.axes_fontsize = 14.3
#g.settings.legend_fontsize = 14.5
g.triangle_plot([samples], names, title_limit=1, filled=True,
                     markers={'\sigma':params[ind,0],'\lambda':params[ind,1]},
                     marker_args={'c':'black'})

plt.show()
plt.savefig('cambridge_2000x20_2xfilter101_order_3_lo.pdf')

Removed no burn in


In [133]:
params2 = np.zeros((len(val_loader),2))
params3 = np.zeros((len(val_loader3),2))
for i, (param, observation, _) in enumerate(val_loader):
    params2[i] = param.cpu()

for i, (param, observation, _) in enumerate(val_loader3):
    params3[i] = param.cpu()

In [166]:
len(np.unique(params3,axis=0))

1572

In [203]:
# here I just try to get an example of a shared parameter
num = 0
shared_params = []
for i, p2 in enumerate(params2):
    same = np.where((p2==params3).all(axis=1),True,False)
    if same.any():
        #now ceck if x shape matches between val graph and suite graph
        num += 1
        index = np.where(same)[0]
        print(p2)
        print(params3[index])
        shared_params.append(params3[index])
    if i ==12: break

#print(shared_params)
#shared_params = np.reshape(shared_params,(-1,2))
#print(num)
#print(len(val_loader))
#print(shared_params)

[2.01010108 7.5454545 ]
[[2.01010108 7.5454545 ]
 [2.01010108 7.5454545 ]]
[44.93939209 18.63636398]
[[44.93939209 18.63636398]
 [44.93939209 18.63636398]
 [44.93939209 18.63636398]
 [44.93939209 18.63636398]]
[ 4.53535366 10.27272701]
[[ 4.53535366 10.27272701]
 [ 4.53535366 10.27272701]]
[34.83838272 12.81818199]
[[34.83838272 12.81818199]
 [34.83838272 12.81818199]]


In [204]:
#shared_param_example = np.array([2.01010108, 7.5454545],dtype='float32')
shared_param_example = np.array([4.53535366, 10.27272701],dtype='float32')
for p2, observation2, _ in val_loader:
    if (p2.cpu()[0].numpy() == shared_param_example).all():
        print(p2)
        #print(p2.cpu()[0])
        break
for p3, observation3, _ in val_loader3:
    if (p3.cpu()[0].numpy() == shared_param_example).all():
        print(p3)
        break

tensor([[ 4.5354, 10.2727]], device='cuda:0')
tensor([[ 4.5354, 10.2727]], device='cuda:0')


In [205]:
posterior = inference.build_posterior(deepcopy(density_estimator))
posterior_samples2 = posterior.sample((10000000,), x=observation2).cpu()
posterior = inference.build_posterior(deepcopy(density_estimator3))
posterior_samples3 = posterior.sample((10000000,), x=observation3).cpu()

names = np.array(['\sigma','\lambda'])


samples2 = MCSamples(samples=posterior_samples2.numpy(), names=names, labels=names, label='$\mathrm{2nd\ order}$')
samples3 = MCSamples(samples=posterior_samples3.numpy(), names=names, labels=names, label='$\mathrm{3rd\ order}$')
g = plots.get_subplot_plotter()
#g.settings.lab_fontsize = 17
#g.settings.axes_fontsize = 14.3
#g.settings.legend_fontsize = 14.5
g.triangle_plot([samples2,samples3], names, title_limit=0, filled=True,
                     markers={'\sigma':shared_param_example[0],'\lambda':shared_param_example[1]},
                     marker_args={'c':'black'})

plt.show()
plt.savefig('cambridge_comparison_2v3_2.pdf')

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


Drawing 10000000 posterior samples:   0%|          | 0/10000000 [00:00<?, ?it/s]

Drawing 10000000 posterior samples:   0%|          | 0/10000000 [00:00<?, ?it/s]

Removed no burn in
Removed no burn in
